# Git 第2周:分支与合并 — Git 的灵魂

> **学习目标**:熟练掌握 merge / rebase / cherry-pick,知道各自适用场景,遇到冲突不慌

---

## Day 8:merge 的三条路径

Git 在执行 `git merge` 时,会根据两个分支的关系自动选择策略:

| 类型 | 条件 | 结果 |
|------|------|------|
| **Fast-forward** | 目标分支是当前分支的直接后代 | 指针前移,无额外 commit |
| **3-way merge** | 两个分支分叉后各有提交 | 生成一个 merge commit |
| **Conflict** | 同一文件同一位置被不同修改 | 需要手动解决冲突 |

### Fast-forward 场景

```
Before merge:           After merge (fast-forward):
main ●──●               main ●──●──●──●
          \                        ↑
feature    ●──●           feature   ●──●
```
两个分支指针最终指向同一个 commit。

In [ ]:
# 模拟 fast-forward 的条件判断

def can_fast_forward(main_commits, feature_commits):
    return feature_commits[0] in main_commits

# 场景1:可以 fast-forward
main = ["A", "B"]
feature = ["B", "C", "D"]
print(f"场景1: main={main}, feature={feature}")
print(f"  fast-forward: {can_fast_forward(main, feature)}")
print("  结果:main 指针直接移动到 D,无 merge commit")

print()

# 场景2:不能 fast-forward(分叉了)
main2 = ["A", "B", "E"]  # main 上也有新 commit
feature2 = ["B", "C", "D"]
print(f"场景2: main={main2}, feature={feature2}")
print(f"  fast-forward: {can_fast_forward(main2, feature2)}")
print("  结果:需要 3-way merge,生成 merge commit")

print()
print("关键:只要 main 在 feature 创建后没有新 commit,就能 fast-forward")

## Day 9:冲突解决

### 冲突标记的含义

```
<<<<<<< HEAD           ← 当前分支(你正在上面的分支)的内容
当前分支的代码
=======                 ← 分界线
被合并分支的代码
>>>>>>> feature        ← 被合并进来的分支名
```

### 配置 diff3 风格(推荐)

```bash
git config --global merge.conflictstyle diff3
```

这样会额外显示共同祖先的内容,更容易判断应该保留哪个。

In [ ]:
# 模拟 merge.conflictstyle=diff3 的冲突展示

base_code  = "timeout = 30                     # 共同祖先"
ours_code  = "timeout = 60  # 本地改为 60s    # HEAD"
theirs_code = "timeout = 90  # 对方改为 90s   # feature"

print(f"文件: config.py")
print(f"<<<<<<< HEAD (当前分支)")
print(ours_code)
print(f"||||||| merged common ancestors (共同祖先)")
print(base_code)
print("=======")
print(theirs_code)
print(f">>>>>>> feature (被合并的分支)")
print()
print("有了 base 版本,你可以看到原本是什么,更容易判断:")
print("- 是对方改了这个逻辑,还是你改了这个逻辑？")
print("- 还是两边都改了？谁的修改更合理？")

## Day 10:rebase — 重写历史

### merge vs rebase 的选择

| 场景 | 推荐 | 原因 |
|------|------|------|
| 把 feature 合并到 main | `git merge` | 保留完整历史,可追溯 |
| feature 分支同步 main 最新代码 | `git rebase main` | 保持 feature 历史干净 |
| 推送前整理本地 commit | `git rebase -i` | squash 零碎 commit |
| 已经 push 的 commit | **不要 rebase!** | 会改写历史,影响他人 |

### rebase 的黄金法则

**永远不要 rebase 已经 push 到共享仓库的 commit。**

In [ ]:
# 模拟 rebase 过程

print("初始状态:")
print("  main:    A──B──C")
print("  feature: A──B──D──E")
print()
print("执行: git checkout feature && git rebase main")
print()
print("Git 做的事:")
print("  1. 找到 feature 和 main 的共同祖先 (B)")
print("  2. 把 B..feature 的 commit (D, E) 暂存为 patch")
print("  3. 把 feature 的 HEAD 移到 main 的最新 commit (C)")
print("  4. 逐个应用 patch (D', E')")
print()
print("rebase 后的历史:")
print("  main:    A──B──C")
print("  feature:          D'──E'   (D'和E'是全新的 commit,SHA 变了)")
print()
print("关键点:D' 和 E' 是全新的 commit(SHA 不同),即使改动一样。")
print("这就是为什么 rebase 后 --force-with-lease 是必须的。")

## Day 11:交互式 rebase 实战

### 六个核心操作

| 命令 | 效果 |
|------|------|
| `pick` | 保留该 commit(默认) |
| `reword` | 修改 commit message |
| `squash` | 合并到上一个 commit,保留 message |
| `fixup` | 合并到上一个 commit,丢弃 message |
| `drop` | 删除该 commit |
| `edit` | 暂停以修改 commit 内容 |

## Day 12:cherry-pick & revert

| 命令 | 用途 | 是否改变历史 |
|------|------|------------|
| `git cherry-pick <sha>` | 把某个 commit 的改动"摘"到当前分支 | 不改变(新增 commit) |
| `git revert <sha>` | 创建一个新 commit 来撤销指定 commit | 不改变(新增 commit) |
| `git reset --hard <sha>` | 强制回到某个版本 | **改变历史!** |

### 典型场景

你在 feature 分支修了一个 bug。这个修复也应该应用到 main 和 release 分支。用 cherry-pick 把那个 commit 精确地"摘"过去,只带这个修复,不带 feature 分支上其他的改动。

## Day 13:reflog — 你的后悔药

reflog 记录 HEAD 和分支引用的每一次移动。默认保留 90 天。

```bash
git reflog                  # 看 HEAD 的移动记录
git reflog show main        # 看 main 分支的移动记录
```

### reflog 能救回来的场景
- `git reset --hard` 后找回"丢失"的 commit
- 误删分支后恢复
- `git commit --amend` 后找回原来的 commit

In [ ]:
import subprocess, os

os.chdir('/home/oa/utils/devops-study')

result = subprocess.run("git reflog -10", shell=True, capture_output=True, text=True)
print("最近 10 条 reflog(HEAD 的移动记录):")
print(result.stdout)
print("\nreflog 的每条记录格式: <sha> HEAD@{n}: <action>: <message>")
print()
print("常见恢复操作:")
print("  git reflog                         # 找到丢失的 SHA")
print("  git reset --hard <sha>             # 恢复到该状态")
print("  git checkout -b recovered <sha>    # 或创建新分支指向该状态")
print()
print("reflog 是 Git 最后的安全网 —— 只要操作提交过,就能找回。")

## Day 14:第2周综合练习

### 分支操作决策矩阵

In [ ]:
scenarios = [
    ("feature 同步 main 最新代码", "rebase", "保持 feature 历史线性干净"),
    ("完成的 feature 合并到 main", "merge(取决于团队规范)", "GitHub Flow: squash; Git Flow: merge commit"),
    ("一个 bugfix 需要应用到两个分支", "cherry-pick", "只需要一个 commit,更精确"),
    ("回滚一个已上线的功能", "revert", "不破坏历史,安全"),
    ("修正上一次 commit 的 typo", "amend + force-push", "仅当分支只有你在用时"),
]

print("场景 → 方案 → 原因")
for s, best, why in scenarios:
    print(f"\n  {s}")
    print(f"  推荐: {best}")
    print(f"  原因: {why}")

print()
print("=" * 60)
print("第2周核心收获:")
print("1. merge 三种情况:fast-forward / 3-way merge / conflict")
print("2. rebase = 重写历史,保持线性;rebase 过的分支必须 force push")
print("3. cherry-pick = 精确摘取单个 commit 的改动")
print("4. revert > reset 在共享分支上(不破坏历史)")
print("5. reflog 是安全网,几乎所有误操作都能找回")
print("=" * 60)